In [ ]:
# import current working diretories

import os
import sys
print(os.getcwd())

In [ ]:
# system path

sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction")

In [ ]:
# present working directory

%pwd

In [ ]:
# changing to the parent directory

os.chdir("../") 

In [ ]:
# present working directory

%pwd

In [ ]:
# import box versions

import box
print(box.__version__)

In [ ]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PredictionPipelineConfig:

    model_path: Path
    preprocessor_unscaled_path: Path

In [ ]:
from Hotel_Booking_Cancellation_Prediction.constant import *
from Hotel_Booking_Cancellation_Prediction.utils.common import read_yaml, create_directories
from Hotel_Booking_Cancellation_Prediction.entity.config_entity import PredictionPipelineConfig
from Hotel_Booking_Cancellation_Prediction.config.configuration import ConfigurationManager

In [ ]:
# configuration manager

def get_prediction_pipeline_config(self) -> PredictionPipelineConfig:

    config = self.config["prediction_pipeline"]

    prediction_pipeline_config = PredictionPipelineConfig(

        model_path=Path(config.model_path),

        preprocessor_unscaled_path =Path(config.preprocessor_path)
    )

    return prediction_pipeline_config

In [ ]:
# components

import pandas as pd
import joblib


class PredictionPipeline:

    def __init__(self, config):

        self.config = config

    # LOAD MODEL

    def load_model(self):

        model = joblib.load(self.config.model_path)

        return model

    # LOAD PREPROCESSOR

    def load_preprocessor(self):

        preprocessor = joblib.load(
            self.config.preprocessor_unscaled_path
        )

        return preprocessor

    # MAKE PREDICTION

    def predict(self, input_data):

        # Load trained model

        model = self.load_model()

        # Load saved preprocessor

        preprocessor = self.load_preprocessor()

        # Convert user input into DataFrame

        input_df = pd.DataFrame([input_data])

        # Apply the SAME preprocessing used during model training i.e., unscaled data

        transformed_input = (
            preprocessor.transform(
                input_df
            )
        )

        # Make prediction

        prediction = model.predict(
            transformed_input)[0]

        # Get prediction probability

        if hasattr(
            model,
            "predict_proba"):

            probability = model.predict_proba(
                transformed_input
            )[0][1]

        else:

            probability = None

        return prediction, probability

In [ ]:
from Hotel_Booking_Cancellation_Prediction.logging import logger

In [ ]:
# pipeline

class PredictionPipelineTrainingPipeline:

    def __init__(self):
        pass

    def main(self,input_data):

        try:

            logger.info(">>>>>> Prediction Pipeline Stage Started <<<<<<")

            config = ConfigurationManager()

            prediction_pipeline_config = (config.get_prediction_pipeline_config() )

            prediction_pipeline = PredictionPipeline(config=prediction_pipeline_config)

            prediction, probability = (prediction_pipeline.predict(input_data))

            logger.info(">>>>>> Prediction Pipeline Stage Completed <<<<<<")

            return prediction, probability


        except Exception as e:

            logger.exception(e)
            raise e

In [ ]:
obj = PredictionPipelineTrainingPipeline()
obj.main()